In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

BASE = "/kaggle/input/notebooks/krishnaiiith/02-feature-extraction"
BASE2 = "/kaggle/input/notebooks/krishnaiiith/01-dataset-preparation"

# CLIP embeddings
text_emb = np.load(BASE + "/text_emb.npy")
img_emb = np.load(BASE + "/img_emb.npy")
hash_emb = np.load(BASE + "/hash_emb.npy")

# hashtag statistics
hashtag_count = np.load(BASE + "/hashtag_count.npy")
hashtag_freq = np.load(BASE + "/hashtag_freq.npy")

# semantic features from notebook 2
semantic_features = np.load(BASE + "/semantic_features.npy")

# labels
df = pd.read_csv(BASE2 + "/labeled.csv")
y = df["label"].values

print("Loaded features:")
print(text_emb.shape, img_emb.shape, hash_emb.shape)

Loaded features:
(11902, 512) (11902, 512) (11902, 512)


In [3]:
def normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)

text_n = normalize(text_emb)
img_n = normalize(img_emb)
hash_n = normalize(hash_emb)

In [4]:
text_img_sim = np.sum(text_n * img_n, axis=1).reshape(-1,1)
text_hash_sim = np.sum(text_n * hash_n, axis=1).reshape(-1,1)
img_hash_sim = np.sum(img_n * hash_n, axis=1).reshape(-1,1)

print("Similarity features:", text_img_sim.shape)

Similarity features: (11902, 1)


In [5]:
text_img_diff = np.abs(text_emb - img_emb)
text_hash_diff = np.abs(text_emb - hash_emb)
img_hash_diff = np.abs(img_emb - hash_emb)

In [6]:
text_img_sim = text_img_sim.reshape(-1,1)
text_hash_sim = text_hash_sim.reshape(-1,1)
img_hash_sim = img_hash_sim.reshape(-1,1)

count = hashtag_count.reshape(-1,1)
freq = hashtag_freq.reshape(-1,1)

In [7]:
import xgboost as xgb
print(xgb.__version__)

3.1.3


In [8]:
X = np.concatenate([
    text_emb,
    img_emb,
    hash_emb,
    text_img_diff,
    text_hash_diff,
    img_hash_diff,
    text_img_sim,
    text_hash_sim,
    img_hash_sim,
    count,
    freq
], axis=1)

print("Final feature shape:", X.shape)

Final feature shape: (11902, 3077)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

pos_weight = neg / (pos + 1e-8)

print("Negative samples:", neg)
print("Positive samples:", pos)
print("scale_pos_weight:", pos_weight)

Negative samples: 8328
Positive samples: 1193
scale_pos_weight: 6.980720871693372


In [12]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [13]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=-1,
              num_parallel_tree=None, ...)

In [14]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("==== MODEL PERFORMANCE ====")
print(classification_report(y_test, y_pred))

roc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", roc)

==== MODEL PERFORMANCE ====
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      2083
           1       0.56      0.37      0.44       298

    accuracy                           0.88      2381
   macro avg       0.74      0.66      0.69      2381
weighted avg       0.87      0.88      0.87      2381

ROC-AUC: 0.8230989441532119


In [15]:
np.save("/kaggle/working/multimodal_features.npy", X)
np.save("/kaggle/working/multimodal_labels.npy", y)

In [16]:
import joblib

joblib.dump(model, "/kaggle/working/xgb_multimodal_model.pkl")

print("Model saved successfully")

Model saved successfully


In [17]:
predictions = pd.DataFrame({
    "true_label": y_test,
    "pred_label": y_pred,
    "probability": y_prob
})

predictions.to_csv("/kaggle/working/test_predictions.csv", index=False)

print("Predictions saved")

Predictions saved
